# WP4v3 — Notebook 1 : Génération de la base de données

Paires : `(cls_mae_raw, cls_clip)` — données **non normalisées** sauvegardées.
Les stats de normalisation sont calculées ici sur le train et sauvegardées séparément.

Deux sets de stats :
- `cls_norm_stats.pt` : pour les CLS tokens (utilisé à l'entraînement)
- `patch_norm_stats.pt` : pour les patch tokens (utilisé à l'inférence)

In [ ]:
import torch
import torch.nn.functional as F
from transformers import ViTImageProcessor, ViTMAEModel
from transformers import LlavaForConditionalGeneration, CLIPImageProcessor
from datasets import load_dataset
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from tqdm import tqdm
import numpy as np

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device : {DEVICE}')


In [ ]:
mae_processor = ViTImageProcessor(
    size={'height': 224, 'width': 224},
    image_mean=[0.485, 0.456, 0.406],
    image_std=[0.229, 0.224, 0.225],
)
mae_encoder = ViTMAEModel.from_pretrained('./vit-mae-large').to(DEVICE)
mae_encoder.eval()

llava = LlavaForConditionalGeneration.from_pretrained(
    './llava-1.5-7b-hf', torch_dtype=torch.float16
)
vision_tower = llava.vision_tower.to(DEVICE).eval()
del llava; torch.cuda.empty_cache()
clip_proc = CLIPImageProcessor.from_pretrained('./llava-1.5-7b-hf')
print(f'Modeles charges — VRAM : {torch.cuda.memory_allocated()/1e9:.1f} GB')


In [ ]:
ds_train = load_dataset('parquet', data_files={'train': './imagenet100/data/train-*.parquet'})
ds_val   = load_dataset('parquet', data_files={'validation': './imagenet100/data/validation-*.parquet'})

transform = transforms.Compose([
    transforms.Resize(384), transforms.CenterCrop(336), transforms.ToTensor(),
])

class HFImageDataset(Dataset):
    def __init__(self, hf_dataset, transform=None):
        self.data = hf_dataset; self.transform = transform
    def __len__(self): return len(self.data)
    def __getitem__(self, idx):
        item = self.data[idx]
        img  = item['image'].convert('RGB')
        if self.transform: img = self.transform(img)
        return img, item['label']

dataset_train = HFImageDataset(ds_train['train'],    transform=transform)
dataset_val   = HFImageDataset(ds_val['validation'], transform=transform)
print(f'Train : {len(dataset_train)} | Val : {len(dataset_val)}')


In [ ]:
def generate_pairs(dataset, batch_size=64, desc=''):
    """
    Retourne des donnees BRUTES (non normalisees) :
    - cls_mae  : token CLS MAE (1024 dim), non normalise
    - cls_clip : token CLS CLIP (1024 dim), L2-normalise
    La normalisation sera appliquee dans le notebook 2.
    """
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=False,
                        num_workers=0, pin_memory=(DEVICE=='cuda'))
    all_cls_mae, all_cls_clip, all_labels = [], [], []

    for images_336, labels in tqdm(loader, desc=desc):
        B = images_336.shape[0]
        images_224 = F.interpolate(images_336, size=(224,224), mode='bilinear', align_corners=False)

        mae_in = mae_processor(images=list(images_224), return_tensors='pt', do_rescale=False)
        mae_in = {k: v.to(DEVICE) for k, v in mae_in.items()}
        with torch.no_grad():
            out = mae_encoder(**mae_in, noise=torch.zeros(B, 196).to(DEVICE))
        cls_mae = out.last_hidden_state[:, 0].cpu().float()    # (B, 1024) brut

        clip_in = clip_proc(images=list(images_336), return_tensors='pt', do_rescale=False)
        pix = clip_in['pixel_values'].to(DEVICE).half()
        with torch.no_grad():
            cls = vision_tower(pix).last_hidden_state[:, 0]    # (B, 1024)
        cls_clip = F.normalize(cls.cpu().float(), dim=-1)       # L2-normalise

        all_cls_mae.append(cls_mae)
        all_cls_clip.append(cls_clip)
        all_labels.append(labels)

    return {
        'cls_mae':  torch.cat(all_cls_mae),   # brut, non normalise
        'cls_clip': torch.cat(all_cls_clip),  # L2-normalise
        'labels':   torch.cat(all_labels),
    }

data_train = generate_pairs(dataset_train, batch_size=64, desc='Train')
data_val   = generate_pairs(dataset_val,   batch_size=64, desc='Val')

torch.save(data_train, 'wp4v3_pairs_train.pt')
torch.save(data_val,   'wp4v3_pairs_val.pt')
print('Sauvegarde OK (donnees brutes)')


In [ ]:
# Stats de normalisation des CLS tokens (sur train uniquement)
cls_mean = data_train['cls_mae'].mean(dim=0)              # (1024,)
cls_std  = data_train['cls_mae'].std(dim=0).clamp(min=1e-6)
torch.save({'mean': cls_mean, 'std': cls_std}, 'wp4v3_cls_norm_stats.pt')
print('Stats CLS sauvegardees')

# Stats de normalisation des patch tokens (sur echantillon du train)
print('Calcul stats patch tokens...')
all_patch = []
for i in tqdm(range(min(500, len(data_train['cls_mae']))), desc='Patch stats'):
    # Recalculer les patch tokens pour un sous-ensemble
    item      = ds_train['train'][i]
    image_pil = item['image'].convert('RGB')
    image_224 = transforms.Compose([
        transforms.Resize(256), transforms.CenterCrop(224), transforms.ToTensor()
    ])(image_pil)
    mae_in = mae_processor(images=[image_224], return_tensors='pt', do_rescale=False)
    mae_in = {k: v.to(DEVICE) for k, v in mae_in.items()}
    with torch.no_grad():
        out = mae_encoder(**mae_in, noise=torch.zeros(1, 196).to(DEVICE))
    all_patch.append(out.last_hidden_state[0, 1:].cpu().float())  # (196, 1024)

all_patch   = torch.cat(all_patch)  # (500*196, 1024)
patch_mean  = all_patch.mean(dim=0)
patch_std   = all_patch.std(dim=0).clamp(min=1e-6)
torch.save({'mean': patch_mean, 'std': patch_std}, 'wp4v3_patch_norm_stats.pt')
print('Stats patch tokens sauvegardees')

# Verification discriminabilite
idx5 = np.random.choice(len(data_val['labels']), 5, replace=False)
for name, key, normalize in [('cls_mae (brut)', 'cls_mae', True), ('cls_clip', 'cls_clip', False)]:
    vecs = data_val[key][idx5].float()
    vecs = F.normalize(vecs, dim=-1)
    sim  = (vecs @ vecs.T).numpy()
    print(f'\nSimilarites cosinus {name} :')
    print(sim.round(4))

del vision_tower, mae_encoder; torch.cuda.empty_cache()
